# Training e Fine-tuning di YOLOv8 su Dataset Personalizzato

Questo notebook affronta i due approcci principali per addestrare YOLO su un **dataset custom**.
Entrambi gli esempi usano **lo stesso dataset** per rendere il confronto diretto e significativo.

| Approccio | Quando usarlo | Pro | Contro | Backbone pre-addestrato | Classi YOLO originali mantenute |
|---|---|---|---|---|---|
| **Fine-tuning** | Dataset piccolo (<1000 img), dominio simile a foto naturali | Converge in poche decine di epoche, ottimi risultati anche con pochi dati | Richiede comunque dati annotati di qualita | **Si** — i pesi sono inizializzati da COCO (80 classi di oggetti reali) | **No** — le classi COCO vengono sostituite con le nuove classi, MA rimane la conoscenza visiva del backbone pre-addestrato |
| **Training from scratch** | Dataset grande (>5k img), dominio molto diverso (medico, satellitare...) | Nessun bias da COCO, massima flessibilita | Richiede molti piu dati ed epoche; convergenza piu lenta e instabile | **No** — pesi inizializzati casualmente | **No** — le classi COCO vengono sostituite con le nuove classi |

## 0. Setup

In [4]:
import os, sys

if 'google.colab' in sys.modules:
    if not os.path.exists('/content/Corso-Computer-Vision'):
        !git clone https://github.com/SalvoSamba01/Corso-Computer-Vision
    %cd /content/Corso-Computer-Vision
    !pip install -r requirements.txt -q
else:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', '../requirements.txt'], capture_output=True)

In [5]:
import os
import yaml
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont


from ultralytics import YOLO
import cv2

sys.path.append('../utils')
from cv_utils import (disegna_detection,mostra_confronto)

import warnings
warnings.filterwarnings('ignore')

In [3]:
if os.path.exists('/content/Corso-Computer-Vision'):
    BASE_DIR = Path('/content/Corso-Computer-Vision')
else:
    BASE_DIR = Path('..').resolve()

EXPERIMENTS_DIR = BASE_DIR / 'results' / 'yolo_training'
EXPERIMENTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'BASE_DIR        : {BASE_DIR}')
print(f'EXPERIMENTS_DIR : {EXPERIMENTS_DIR}')


BASE_DIR        : C:\Users\smbsv\Desktop\Corso-Computer-Vision
EXPERIMENTS_DIR : C:\Users\smbsv\Desktop\Corso-Computer-Vision\results\yolo_training


---
## 1. Formato delle annotazioni YOLO e struttura del dataset

Prima di addestrare qualunque modello, e fondamentale capire come YOLO si aspetta i dati.
Il tool di annotazione piu comodo per creare/esportare dataset in questo formato e
[Roboflow](https://roboflow.com) (gratuito), ma anche LabelImg, CVAT e Label Studio supportano
l'esportazione in YOLO format.

### 1.1 Struttura delle cartelle

```
mio_dataset/
|-- data.yaml              <- file di configurazione (obbligatorio)
|-- train/
|   |-- images/            <- immagini di training  (.jpg | .png | .bmp ...)
|   +-- labels/            <- annotazioni           (.txt, una per immagine)
|-- valid/
|   |-- images/
|   +-- labels/
+-- test/                  <- opzionale
    |-- images/
    +-- labels/
```

> Le directory `images/` e `labels/` devono essere allo stesso livello e avere lo stesso nome
> di radice. YOLO trova i `.txt` sostituendo `images` con `labels` nel percorso.

### 1.2 File di annotazione `.txt`

Ogni immagine ha un file `.txt` con lo stesso nome base (es. `foto_001.jpg` -> `foto_001.txt`).
Ogni riga del `.txt` descrive **un oggetto**:

```
<class_id> <cx> <cy> <w> <h>
```

| Campo | Tipo | Significato |
|---|---|---|
| `class_id` | intero | Indice della classe (0-based), coerente con `names` in `data.yaml` |
| `cx`, `cy` | float [0,1] | Centro della bounding box **normalizzato** sulla larghezza/altezza dell'immagine |
| `w`, `h` | float [0,1] | Larghezza e altezza della bounding box **normalizzate** |

**Esempio**: oggetto di classe 2, centrato a 30% da sinistra, 50% dall'alto, largo 20%, alto 15%:
```
2 0.300 0.500 0.200 0.150
```

Se un'immagine non contiene oggetti, il file `.txt` esiste ma e **vuoto**
(o puo essere omesso a seconda del tool — meglio tenerlo vuoto).

### 1.3 File `data.yaml`

```yaml
path: /percorso/assoluto/al/dataset   
train: train/images
val:   valid/images
test:  test/images                     # opzionale

nc: 3                                  # numero di classi (number of classes)
names: ['gatto', 'cane', 'persona']    # nomi nell'ordine degli id (0, 1, 2)
```

### 1.4 Data augmentation automatica

YOLOv8 applica augmentation on-the-fly durante il training (flip, mosaic, mixup, HSV jitter...).
Non è necessario pre-aumentare il dataset manualmente.


In [4]:
# ============================================================
#  SETTING
# ============================================================

DATASET_YAML = '../data/day_3/animals_yolo/data.yaml'   # <- MODIFICARE questo percorso per cambiare dataset

# --- Leggi classi dal data.yaml ---
if Path(DATASET_YAML).exists():
    with open(DATASET_YAML) as f:
        data_cfg = yaml.safe_load(f)
    CLASS_NAMES = data_cfg.get('names', [])
    NUM_CLASSES = data_cfg.get('nc', len(CLASS_NAMES))
    print(f'Dataset caricato: {NUM_CLASSES} classi -> {CLASS_NAMES}')
else:
    print(f'[ATTENZIONE] File non trovato: {DATASET_YAML}')
    print('Modifica DATASET_YAML con il percorso corretto al tuo data.yaml.')
    CLASS_NAMES = []
    NUM_CLASSES = 0


Dataset caricato: 5 classi -> ['cat', 'cow', 'dog', 'hen', 'horse']


---
## 2. Fine-tuning di YOLOv8 (Transfer Learning)

Il backbone di YOLOv8 pre-addestrato su COCO ha imparato a riconoscere feature visive
generali: bordi, texture, forme, gradienti. Il fine-tuning sfrutta questa conoscenza
applicandola al tuo problema specifico.

**Come funziona**:
1. I pesi del backbone vengono caricati da COCO
2. L'head di detection viene reinizializzata con `nc` classi del tuo dataset
3. Il training aggiorna tutti i pesi (backbone + head), ma partendo da valori gia buoni

**Risultato**: converge in **20-50 epoche** anche con poche centinaia di immagini.

### Scelta del modello base

| Modello | Parametri | mAP COCO | Velocita (A100) | Quando usarlo |
|---|---|---|---|---|
| `yolov8n.pt` | 3.2M | 37.3 | ~80 FPS | CPU, mobile, edge, prototipazione rapida |
| `yolov8s.pt` | 11.2M | 44.9 | ~50 FPS | **Bilanciato — buon punto di partenza** |
| `yolov8m.pt` | 25.9M | 50.2 | ~25 FPS | GPU consumer, buona accuratezza |
| `yolov8l.pt` | 43.7M | 52.9 | ~15 FPS | Produzione, GPU dedicata |
| `yolov8x.pt` | 68.2M | 53.9 | ~10 FPS | Massima accuratezza, inferenza non real-time |

> Per la demo usiamo `yolov8n` (il piu leggero). Su Colab con GPU passa a `yolov8s` o superiore.


In [5]:
# ============================================================
#  FINE-TUNING -- parametri configurabili
# ============================================================

FT_MODEL      = 'yolov8n.pt'          # modello pre-addestrato (scaricato automaticamente)
FT_DATA       = DATASET_YAML          # stesso dataset usato anche per from scratch
FT_EPOCHS     = 1                    # 30-100 epoche sono tipicamente sufficienti
FT_BATCH      = 16                    # riduci a 8 se vai out-of-memory
FT_PATIENCE   = 20                    # early stopping: stop se mAP non migliora per N epoche
FT_PROJECT    = str(EXPERIMENTS_DIR)
FT_NAME       = 'finetune_run'
FT_PRETRAINED = True                  # <- TRUE: usa i pesi pre-addestrati su COCO

print('Configurazione fine-tuning:')
print(f'  model     : {FT_MODEL}')
print(f'  data      : {FT_DATA}')
print(f'  epochs    : {FT_EPOCHS}')
print(f'  batch     : {FT_BATCH}')
print(f'  patience  : {FT_PATIENCE}')
print(f'  pretrained: {FT_PRETRAINED}')


Configurazione fine-tuning:
  model     : yolov8n.pt
  data      : ../data/day_3/animals_yolo/data.yaml
  epochs    : 1
  batch     : 16
  patience  : 20
  pretrained: True


In [6]:
model_ft = YOLO(FT_MODEL)

print('--- Inizio fine-tuning ---')
print(f'Epoche: {FT_EPOCHS}  |  Dataset: {FT_DATA}')
print('Qualche minuto su CPU, pochi secondi/epoca su GPU.')

results_ft = model_ft.train(
    data       = FT_DATA,
    epochs     = FT_EPOCHS,
    batch      = FT_BATCH,
    patience   = FT_PATIENCE,
    project    = FT_PROJECT,
    name       = FT_NAME,
    exist_ok   = True,
    pretrained = FT_PRETRAINED,
)

print('--- Fine-tuning completato ---')
print(f'Modello salvato in: {results_ft.save_dir}')

--- Inizio fine-tuning ---
Epoche: 1  |  Dataset: ../data/day_3/animals_yolo/data.yaml
Qualche minuto su CPU, pochi secondi/epoca su GPU.
New https://pypi.org/project/ultralytics/8.4.32 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.24  Python-3.12.13 torch-2.10.0+cpu CPU (Intel Core i7-10750H 2.60GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../data/day_3/animals_yolo/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.

### Cosa stiamo misurando: le metriche di YOLO

Prima di visualizzare i grafici, e importante capire cosa rappresenta ogni metrica.

---

#### Loss (funzioni di costo) — valore da minimizzare

| Metrica | Cosa misura |
|---|---|
| **box_loss** | Errore sulla posizione e dimensione delle bounding box (usa DIoU/CIoU loss) |
| **cls_loss** | Errore sulla classificazione dell'oggetto (cross-entropy) |
| **dfl_loss** | Distribution Focal Loss — errore sulla distribuzione della posizione dei bordi |

In un training sano, tutte le loss **scendono** nel tempo.
Se `val_loss` smette di scendere mentre `train_loss` continua -> **overfitting**.

---

#### Metriche di rilevamento — valore da massimizzare

**Precision (P)**: di tutti gli oggetti che il modello ha rilevato, quanti erano reali?
```
P = TP / (TP + FP)
```
Alta Precision = pochi falsi positivi (il modello non "inventa" oggetti).

**Recall (R)**: di tutti gli oggetti reali presenti, quanti ha trovato il modello?
```
R = TP / (TP + FN)
```
Alto Recall = pochi falsi negativi (il modello non "manca" oggetti).

> Precision e Recall sono in trade-off: abbassando la soglia di confidenza aumenta il recall
> ma diminuisce la precision (e viceversa). Per questo si usa la curva PR.

**mAP@50** (mean Average Precision a IoU=0.5): la metrica principale per la detection.
- Per ogni classe calcola l'area sotto la curva Precision-Recall con soglia IoU=0.50
- Poi fa la media su tutte le classi
- IoU=0.50 significa che una predizione e corretta se si sovrappone almeno al 50% con il ground truth

**mAP@50-95**: versione piu rigorosa — media di mAP calcolata per soglie IoU da 0.50 a 0.95 a step di 0.05.
E la metrica ufficiale del benchmark COCO. Un mAP50-95 > 0.50 e considerato molto buono.

---

#### Interpretazione pratica

| Scenario | Sintomo | Causa probabile |
|---|---|---|
| Loss scende, mAP sale | Tutto normale | Training sano |
| `val/box_loss` sale dopo un po' | Overfitting sulle bbox | Troppo poche immagini o troppi epochs |
| `train/cls_loss` non scende | Underfitting | LR troppo bassa, classi mal bilanciate |
| mAP@50 alto, mAP@50-95 basso | Box imprecise | Il modello trova gli oggetti ma le bbox sono approssimative |
| Recall basso, Precision alta | Modello troppo conservativo | Confidence threshold troppo alto |


### Metriche di training

Vengono salvate automaticamente nella cartella del run:
- `results.csv` — loss e metriche per ogni epoca
- `confusion_matrix.png`, `PR_curve.png`, `F1_curve.png` — grafici diagnostici
- `weights/best.pt` — checkpoint con il miglior mAP50-95
- `weights/last.pt` — ultimo checkpoint salvato


### Esempio di detection con il nuovo modello


In [7]:
model = YOLO("../results/yolo_training/finetune_run/weights/best.pt")

img_path = "../data/day_3/animals_yolo/test/images/cat-1-_jpg.rf.6b98ea0c71e3a14b585999b6856f586d.jpg"


# immagine originale (BGR)
img = cv2.imread(img_path)

img_draw = img.copy()

# inferenza
res = model(img_path, conf=0.1)

names = model.names

for box in res[0].boxes:
    x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
    conf = float(box.conf[0].cpu().numpy())
    cls = int(box.cls[0].cpu().numpy())

    label = f"{names[cls]} {conf:.2f}"

    cv2.rectangle(img_draw, (x1, y1), (x2, y2), (0, 255, 0), 2)

    cv2.putText(img_draw, label, (x1, y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)


cv2.imshow("Image", img_draw)
cv2.waitKey(0)
cv2.destroyAllWindows()

FileNotFoundError: [Errno 2] No such file or directory: '../data/day_3/animals_yolo/test/images/cat-1-_jpg.rf.6b98ea0c71e3a14b585999b6856f586d.jpg'

---
## 3. Training From Scratch

Parte da pesi inizializzati **casualmente**: nessuna conoscenza pre-acquisita.
Usato quando il dominio visivo è radicalmente diverso dalle immagini naturali di COCO
(es. microscopia, immagini termiche, radar, raggi X) oppure quando si hanno molti dati.

**Come funziona**:
1. I pesi del backbone vengono inizializzati casualmente (o con tecniche apposite)
2. L'head di detection viene inizializzata casualmente con `nc` classi del tuo dataset
3. Il training deve imparare tutto da zero: feature di basso livello (bordi, texture)
   e di alto livello (forme, oggetti complessi) insieme

**Trade-off**: richiede **più epoche** e **più dati** rispetto al fine-tuning
per raggiungere prestazioni simili. Le prime epoche la loss e molto alta e instabile.

**Stesso dataset del fine-tuning**: confrontare i due approcci sullo stesso dataset permette
di vedere chiaramente il vantaggio del transfer learning in termini di velocita di convergenza
e qualita finale, specialmente con dataset piccoli.


In [ ]:
# ============================================================
#  TRAINING FROM SCRATCH -- parametri configurabili
# ============================================================

# yolov8n.yaml carica SOLO l'architettura (nessun peso pre-addestrato).
# Equivale a: YOLO('yolov8n.pt') con pretrained=False
SC_MODEL         = 'yolov8n.yaml'
SC_DATA          = DATASET_YAML        # stesso dataset del fine-tuning per confronto diretto
SC_EPOCHS        = 100                 # servono molte piu epoche rispetto al fine-tuning
SC_IMGSZ         = 640
SC_BATCH         = 16
SC_LR0           = 0.01
SC_LRF           = 0.001               # lr finale piu bassa: decadimento piu lento
SC_WARMUP_EPOCHS = 10                  # warmup piu lungo: i pesi sono casuali, serve stabilita
SC_PATIENCE      = 30                  # early stopping: piu pazienza, converge piu lentamente
SC_PROJECT       = str(EXPERIMENTS_DIR)
SC_NAME          = 'scratch_run'
SC_PRETRAINED    = False               # <- FALSE: training from scratch

print('Configurazione training from scratch:')
print(f'  model         : {SC_MODEL}')
print(f'  data          : {SC_DATA}')
print(f'  epochs        : {SC_EPOCHS}')
print(f'  imgsz         : {SC_IMGSZ}')
print(f'  batch         : {SC_BATCH}')
print(f'  lr0           : {SC_LR0}')
print(f'  lrf           : {SC_LRF}')
print(f'  warmup_epochs : {SC_WARMUP_EPOCHS}')
print(f'  patience      : {SC_PATIENCE}')
print(f'  pretrained    : {SC_PRETRAINED}')


In [ ]:
model_scratch = YOLO(SC_MODEL)

print('--- Inizio training from scratch ---')
print('I pesi sono casuali: le prime epoche la loss sara alta e instabile.')
print('E normale: il modello deve imparare tutto da zero.')

results_scratch = model_scratch.train(
    data          = SC_DATA,
    epochs        = SC_EPOCHS,
    imgsz         = SC_IMGSZ,
    batch         = SC_BATCH,
    lr0           = SC_LR0,
    lrf           = SC_LRF,
    warmup_epochs = SC_WARMUP_EPOCHS,
    patience      = SC_PATIENCE,
    project       = SC_PROJECT,
    name          = SC_NAME,
    exist_ok      = True,
    pretrained    = SC_PRETRAINED,
)

print('--- Training from scratch completato ---')
print(f'Modello salvato in: {results_scratch.save_dir}')


---
## 4. Confronto: Fine-tuning vs Training From Scratch

Entrambi i training hanno usato lo **stesso dataset** e lo stesso modello base (yolov8n).
Il confronto e diretto: l'unica variabile e il punto di partenza dei pesi.


In [ ]:
def leggi_risultati(save_dir):
    csv_path = Path(save_dir) / 'results.csv'
    if not csv_path.exists():
        return None
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    return df

df_ft      = leggi_risultati(results_ft.save_dir)
df_scratch = leggi_risultati(results_scratch.save_dir)

if df_ft is not None and df_scratch is not None:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for col, ax, title in [
        ('metrics/mAP50',    axes[0], 'mAP@50'),
        ('metrics/mAP50-95', axes[1], 'mAP@50-95'),
        ('val/box_loss',     axes[2], 'Val Box Loss'),
    ]:
        if col in df_ft.columns:
            ax.plot(df_ft['epoch'],      df_ft[col],      label='Fine-tuning',  color='steelblue', linewidth=2)
        if col in df_scratch.columns:
            ax.plot(df_scratch['epoch'], df_scratch[col], label='From Scratch', color='tomato',    linewidth=2, linestyle='--')
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('Epoca')
        ax.legend()
        ax.grid(alpha=0.3)

    plt.suptitle(f'Fine-tuning vs Training From Scratch  |  stesso dataset', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print('\n=== Risultati finali ===')
    print(f'{"Metrica":<25} {"Fine-tuning":>15} {"From Scratch":>15}')
    print('-' * 57)
    for col in ['metrics/precision','metrics/recall','metrics/mAP50','metrics/mAP50-95']:
        nome   = col.split('/')[-1]
        val_ft = df_ft.iloc[-1].get(col, float('nan'))
        val_sc = df_scratch.iloc[-1].get(col, float('nan'))
        winner = '  <- migliore' if val_ft > val_sc else ''
        print(f'{nome:<25} {val_ft:>15.4f} {val_sc:>15.4f}{winner}')
else:
    print('Esegui prima entrambi i training.')


---
## 5. Validazione e Inferenza


In [ ]:
# Valuta il miglior modello fine-tuned sul validation set
best_weights = Path(results_ft.save_dir) / 'weights' / 'best.pt'

if best_weights.exists():
    model_eval = YOLO(str(best_weights))
    metrics    = model_eval.val(data=DATASET_YAML, verbose=True)

    print('\n=== Metriche di validazione (best.pt) ===')
    print(f'  mAP50    : {metrics.box.map50:.4f}')
    print(f'  mAP50-95 : {metrics.box.map:.4f}')
    print(f'  Precision: {metrics.box.mp:.4f}')
    print(f'  Recall   : {metrics.box.mr:.4f}')
else:
    print(f'Pesi non trovati: {best_weights}')


In [ ]:
# Inferenza su immagini del validation set
best_weights = Path(results_ft.save_dir) / 'weights' / 'best.pt'

if not best_weights.exists():
    print('Esegui prima il fine-tuning.')
else:
    model_infer = YOLO(str(best_weights))

    # Cerca immagini dal validation set indicato nel data.yaml
    with open(DATASET_YAML) as f:
        cfg = yaml.safe_load(f)
    base    = Path(cfg.get('path', Path(DATASET_YAML).parent))
    val_dir = base / cfg.get('val', 'valid/images')
    imgs    = sorted(val_dir.glob('*.*'))[:6]

    if imgs:
        res_list = model_infer.predict([str(p) for p in imgs], conf=0.25, verbose=False)
        fig, axes = plt.subplots(2, 3, figsize=(16, 10))
        for ax, res in zip(axes.flatten(), res_list):
            ax.imshow(res.plot()[:,:,::-1])
            ax.axis('off')
            ax.set_title(f'{len(res.boxes)} detection(s)', fontsize=9)
        plt.suptitle('Inferenza - Modello fine-tuned (best.pt)', fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.show()
    else:
        print(f'Nessuna immagine trovata in {val_dir}')
        print('Controlla il campo "val" nel data.yaml.')


---
## 6. Esportazione del modello

Una volta addestrato e validato, esporti il modello nel formato ottimale per il deployment.


In [ ]:
best_weights = Path(results_ft.save_dir) / 'weights' / 'best.pt'

if best_weights.exists():
    model_export = YOLO(str(best_weights))
    # ONNX e il formato piu universale per il deployment cross-platform
    onnx_path    = model_export.export(format='onnx', imgsz=640, simplify=True)
    print(f'Modello esportato: {onnx_path}')

    print('\nAltri formati disponibili:')
    formati = [
        ('onnx',        'ONNX          -- cross-platform, compatibile con TensorRT e OpenVINO'),
        ('torchscript', 'TorchScript   -- deployment PyTorch senza dipendenze Python'),
        ('tflite',      'TFLite        -- Android, microcontrollori, edge devices'),
        ('coreml',      'CoreML        -- iOS e macOS (Apple Silicon)'),
        ('openvino',    'OpenVINO      -- Intel CPU/GPU, ottimizzato per inferenza'),
        ('engine',      'TensorRT      -- NVIDIA GPU, massima velocita su dispositivi CUDA'),
    ]
    for fmt, desc in formati:
        print(f'  model.export(format="{fmt}")  ->  {desc}')
else:
    print('Esegui prima il fine-tuning.')


---
## Riepilogo e Regole Pratiche

### Cheatsheet: le 3 righe fondamentali

**Fine-tuning (transfer learning)**
```python
from ultralytics import YOLO
model = YOLO('yolov8n.pt')                        # backbone COCO pre-addestrato
model.train(data='data.yaml', epochs=50)           # adatta al tuo dataset
```

**Training from scratch**
```python
from ultralytics import YOLO
model = YOLO('yolov8n.yaml')                       # solo architettura, pesi casuali
model.train(data='data.yaml', epochs=150, pretrained=False, warmup_epochs=10)
```

**Valutazione e inferenza**
```python
model   = YOLO('runs/detect/train/weights/best.pt')
metrics = model.val(data='data.yaml')              # mAP, precision, recall
results = model.predict('img.jpg', conf=0.25)      # inferenza su immagine/video/cartella
model.export(format='onnx')                        # esporta per deployment
```

---

### Quale approccio scegliere?

| Situazione | Approccio consigliato | Motivazione |
|---|---|---|
| Dataset < 200 img | Fine-tuning con augmentation aggressiva | Il backbone pre-addestrato compensa la mancanza di dati |
| Dataset 200-2000 img | Fine-tuning | Il punto di partenza dei pesi vale piu dei dati aggiuntivi |
| Dataset > 5000 img, dominio simile a COCO | Fine-tuning | Ancora conveniente, converge prima |
| Dataset > 5000 img, dominio molto diverso | Training from scratch | Il backbone COCO potrebbe introdurre bias non utili |
| Vincoli legali sui pesi | Training from scratch | Impossibile usare pesi COCO se non autorizzati |
| Prototipazione rapida | Fine-tuning con `yolov8n` | Risultati decenti in pochi minuti |

---

### Iperparametri chiave e come regolarli

| Parametro | Default | Quando aumentarlo | Quando diminuirlo |
|---|---|---|---|
| `epochs` | 50 (ft) / 150 (scratch) | Dataset grande, convergenza lenta | Early stopping si attiva presto -> gia abbastanza |
| `batch` | 16 | GPU con molta VRAM (batch 32/64 stabilizza il training) | Out-of-memory |
| `lr0` | 0.01 | Raramente (puo causare instabilita) | Loss oscilla senza scendere |
| `lrf` | 0.01 | Vuoi un lr finale piu basso (decadimento piu forte) | Loss scende troppo lentamente alla fine |
| `patience` | 15-30 | Curva di apprendimento rumorosa (validation loss oscilla) | Vuoi fermare prima |
| `warmup_epochs` | 3 (ft) / 10 (scratch) | From scratch o lr0 alta | Fine-tuning con lr bassa |
| `imgsz` | 640 | Oggetti piccoli (provare 1280) | Memoria insufficiente (provare 320/416) |

---

### Qualita dei dati: quello che conta davvero

Il modello non puo essere migliore dei dati su cui e addestrato.

**Bilanciamento delle classi**: se una classe ha 10x piu esempi delle altre, il modello
la privilegera. Soluzioni: sotto-campionare la classe maggioritaria, sovra-campionare la
minoranza, o pesare diversamente le classi all'interno della loss, nel training.

**Qualita delle annotazioni**: bounding box imprecise o etichette errate sono il problema piu
comune.

**Varieta delle immagini**: angolazioni, illuminazioni, sfondi e scale diverse migliorano
la generalizzazione.

**Train/Val/Test split**: tipicamente 70/20/10. Il validation set deve essere rappresentativo
ma non deve "trapelare" nel training (meglio evitare di usare le stesse immagini aumentate).

---

### Sintomi e diagnosi

| Sintomo osservato | Diagnosi | Soluzione |
|---|---|---|
| `val/loss` sale mentre `train/loss` scende | Overfitting | Piu dati, augmentation piu forte, early stopping, dropout |
| Entrambe le loss rimangono alte | Underfitting | LR troppo bassa, pochi epochs, architettura troppo piccola |
| mAP@50 alto ma mAP@50-95 basso | Bbox imprecise | Aumenta `epochs`, usa modello piu grande, controlla annotazioni |
| Recall basso, Precision alta | Modello troppo conservativo | Abbassa soglia di confidenza (`conf=0.15`), controlla classi sbilanciate |
| Loss instabile (zig-zag) | LR troppo alta o batch troppo piccolo | Riduci `lr0`, aumenta `batch` |
| Training from scratch non converge | Dataset troppo piccolo | Passa al fine-tuning, aggiungi dati, aumenta `warmup_epochs` |

---

### Struttura dei file dopo il training

```
results/yolo_training/
|-- finetune_run/
|   |-- weights/
|   |   |-- best.pt       <- usa questo per inferenza e export
|   |   +-- last.pt       <- ultimo checkpoint (utile per riprendere il training)
|   |-- results.csv       <- metriche per epoca
|   |-- args.yaml         <- tutti gli iperparametri usati (riproducibilita)
|   |-- confusion_matrix.png
|   |-- PR_curve.png
|   |-- F1_curve.png
|   +-- val_batch0_pred.jpg
+-- scratch_run/
    +-- ...
```

> Per riprendere un training interrotto: `model.train(resume=True)` — usa automaticamente
> `last.pt` e gli iperparametri salvati in `args.yaml`.
